# Inter-Annotator Agreement (IAA) Analysis


## Round 1

Measures agreement between the three annotators (**MC**, **ME**, **TC**) who screened a
sample of papers for inclusion, and merges their verdicts into a single table.

The screening subsets were produced in [02_data_merge.ipynb](02_data_merge.ipynb). The IAA
design is:

- **20 papers** reviewed by **all 3** annotators (`iaa_set == "all"`)
- **10 papers per pair** reviewed by exactly 2 annotators (`iaa_set == "pair_XX_YY"`)

So every **pair shares 30 papers** (20 + 10), all three share **20**, and there are **50
unique papers** total (20 by all 3, 30 by exactly 2).

The variable of interest is **`Include`** (boolean): whether the annotator would include
the paper. The completed annotations live in `../data/processed/iaa/annotated/`.


In [16]:
import sys
sys.path.append("../")  # go up one level from notebooks/ to project root

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import collections
import krippendorff
from sklearn.metrics import cohen_kappa_score

ANNOTATORS = ["MC", "ME", "TC"]
# Per-annotator columns (vary by annotator); "Include" is the variable of interest.
ANNOTATION_COLS = ["Include", "To discuss", "Research area", "Topic/Ideas", "Notes"]
# Paper-level columns (identical across annotators for a given paper).
META_COLS = ["iaa_set", "ids", "year", "sources", "title", "abstract",
             "published_in", "paper_type"]

iaa_dir = Path("../data/processed/iaa/annotated/round1")

## Load annotated files

In [2]:
dfs = {a: pd.read_excel(iaa_dir / f"annotator_{a}_annotated.xlsx") for a in ANNOTATORS}

for a, df in dfs.items():
    assert len(df) == 40, f"{a} has {len(df)} rows (expected 40)"
    assert df["Include"].notna().all(), f"{a} has missing Include values"
    print(f"Annotator {a}: {len(df)} rows  "
          f"iaa_set={df['iaa_set'].value_counts().to_dict()}")

Annotator MC: 40 rows  iaa_set={'all': 20, 'pair_MC_ME': 10, 'pair_MC_TC': 10}
Annotator ME: 40 rows  iaa_set={'all': 20, 'pair_MC_ME': 10, 'pair_ME_TC': 10}
Annotator TC: 40 rows  iaa_set={'all': 20, 'pair_ME_TC': 10, 'pair_MC_TC': 10}


## Wide `Include` matrix

One row per paper, one column per annotator. `NaN` means the annotator did not review that
paper. This backs the agreement metrics and the derived counts.

In [3]:
long = pd.concat(
    [df[["paper_id", "Include"]].assign(annotator=a) for a, df in dfs.items()],
    ignore_index=True,
)
include_wide = long.pivot(index="paper_id", columns="annotator", values="Include")[ANNOTATORS]
print(f"Unique papers reviewed: {len(include_wide)}")
include_wide.head()

Unique papers reviewed: 50


annotator,MC,ME,TC
paper_id,,,
031fcedf9970f53c,False,False,NaN
06e5f45d08bfd760,NaN,True,False
0eea50f3d58ec619,False,False,False
1aecf98d861b4aea,NaN,True,False
225076b0055d9170,False,False,False


## 1. Cohen's kappa per pair of annotators

Each pair shares **30** papers (20 reviewed by all + 10 pairwise). 

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ the agreement expected by chance from the annotators' marginal label frequencies.

Our results show fair agreement in general.

In [4]:
kappa_rows = []
for x, y in combinations(ANNOTATORS, 2):
    both = include_wide[[x, y]].dropna()  # the 30 papers reviewed by both
    k = cohen_kappa_score(both[x].astype(int), both[y].astype(int))
    kappa_rows.append({"pair": f"{x}-{y}", "n_shared": len(both), "cohen_kappa": k})

kappa_df = pd.DataFrame(kappa_rows).round(3)
print(kappa_df[["pair", "cohen_kappa"]])
kappa_df


    pair  cohen_kappa
0  MC-ME        0.384
1  MC-TC        0.348
2  ME-TC        0.342


,pair,n_shared,cohen_kappa
0,MC-ME,30,0.384
1,MC-TC,30,0.348
2,ME-TC,30,0.342


## 2. Krippendorff's alpha (3 annotators)

Computed on the **20 papers** reviewed by all three annotators. The reliability matrix has shape `3 raters × 20 units`.

In [5]:
all_papers = include_wide.dropna()  # 20 papers reviewed by all three
reliability = all_papers[ANNOTATORS].astype(int).T.values  # 3 x 20

alpha = krippendorff.alpha(reliability_data=reliability, level_of_measurement="nominal")
print(f"Papers reviewed by all three: {len(all_papers)}")
print(f"Krippendorff's alpha (nominal): {alpha:.3f}")

Papers reviewed by all three: 20
Krippendorff's alpha (nominal): 0.454


## 3. Merged annotations table

One row per paper (50 total), carrying the paper metadata once plus every annotator's
`Include`, `To discuss`, `Research area`, `Topic/Ideas` and `Notes` (suffixed by annotator
code). `NaN` in an annotator's columns means they did not review that paper.

Derived columns:
- `n_reviewers` — how many annotators reviewed the paper,
- `n_include` — how many marked it for inclusion,
- `disagreement` — `True` when the reviewers are **not unanimous** (at least two disagree),
- `any_to_discuss` — `True` when at least one reviewer flagged the paper "To discuss".

In [6]:
# Paper-level columns once (identical across annotators for a given paper).
meta = (pd.concat([df[["paper_id"] + META_COLS] for df in dfs.values()], ignore_index=True)
        .drop_duplicates(subset="paper_id"))

# Per-annotator annotation columns, suffixed by annotator code.
ann_frames = [
    df[["paper_id"] + ANNOTATION_COLS]
    .rename(columns={c: f"{a}_{c}" for c in ANNOTATION_COLS})
    .set_index("paper_id")
    for a, df in dfs.items()
]
ann_wide = ann_frames[0].join(ann_frames[1:], how="outer")

merged = meta.set_index("paper_id").join(ann_wide, how="outer").reset_index()

# Derived columns from the per-annotator Include verdicts.
inc_cols = [f"{a}_Include" for a in ANNOTATORS]
merged["n_reviewers"] = merged[inc_cols].notna().sum(axis=1)
merged["n_include"] = (merged[inc_cols] == True).sum(axis=1)
merged["disagreement"] = (merged["n_include"] > 0) & (merged["n_include"] < merged["n_reviewers"])

# True if at least one reviewer flagged the paper "To discuss".
disc_cols = [f"{a}_To discuss" for a in ANNOTATORS]
merged["any_to_discuss"] = (merged[disc_cols] == True).any(axis=1)

# Column order: identifiers, per-annotator Include, derived flags, rest of metadata, notes.
note_cols = [f"{a}_{c}" for c in ANNOTATION_COLS if c != "Include" for a in ANNOTATORS]
# ordered = (["paper_id", "title", "year"] 
ordered = (["paper_id"] 
        #    + [c for c in META_COLS if c not in ("title", "year")] 
           + META_COLS
           + inc_cols
           + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]
           + note_cols)
merged = merged[ordered]

print(f"Merged table: {merged.shape[0]} papers x {merged.shape[1]} columns")
merged[["paper_id", "title"] + inc_cols
       + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]]

Merged table: 50 papers x 28 columns


,paper_id,title,MC_Include,ME_Include,TC_Include,n_reviewers,n_include,disagreement,any_to_discuss
0,031fcedf9970f53c,25th European Conference on Advances in Databa...,False,False,NaN,2,0,False,True
1,06e5f45d08bfd760,Social media discourse as a window into energy...,NaN,True,False,2,1,True,False
2,0eea50f3d58ec619,Where did you tweet from? Inferring the origin...,False,False,False,3,0,False,True
3,1aecf98d861b4aea,Navigating the Map of Environmental-policy Nar...,NaN,True,False,2,1,True,False
4,225076b0055d9170,Estimation of corn yield using multi-temporal ...,False,False,False,3,0,False,False
5,27181efdda96f11a,SEN12-WATER: A New Dataset for Hydrological Ap...,False,False,NaN,2,0,False,False
6,278d67b0621a5c36,The Presence of Digital Volunteer and The Abse...,NaN,False,False,2,0,False,True
7,295dd9e3895f13f1,Opinion-based homogeneity on youtube: Combinin...,False,True,True,3,2,True,True
8,2cb4c1da63ae4b99,Navigating the winds of change: strategic fore...,False,False,False,3,0,False,False
9,2d9dff826cc1e5bd,Climate Change Myths Detection Using Dynamical...,True,True,NaN,2,2,False,False


In [7]:
# column names
merged.columns

Index(['paper_id', 'iaa_set', 'ids', 'year', 'sources', 'title', 'abstract',
       'published_in', 'paper_type', 'MC_Include', 'ME_Include', 'TC_Include',
       'n_reviewers', 'n_include', 'disagreement', 'any_to_discuss',
       'MC_To discuss', 'ME_To discuss', 'TC_To discuss', 'MC_Research area',
       'ME_Research area', 'TC_Research area', 'MC_Topic/Ideas',
       'ME_Topic/Ideas', 'TC_Topic/Ideas', 'MC_Notes', 'ME_Notes', 'TC_Notes'],
      dtype='str')

In [8]:
output_path = iaa_dir / "iaa_merged.xlsx"
merged.to_excel(output_path, index=False)
print(f"Wrote {output_path}")

Wrote ../data/processed/iaa/annotated/round1/iaa_merged.xlsx


## 4. Basic statistics

In [9]:
n_total = len(merged)
n_any_include = int((merged["n_include"] >= 1).sum())
n_unanimous_include = int((merged["n_include"] == merged["n_reviewers"]).sum())
n_disagreement = int(merged["disagreement"].sum())
n_unanimous_exclude = int((merged["n_include"] == 0).sum())

print(f"Total unique papers reviewed                        : {n_total}")
print(f"Papers with at least one include                    : {n_any_include}")
print(f"Papers included by all reviewers who checked it     : {n_unanimous_include}")
print(f"Soft acceptance rate (at least one include / total) : {n_any_include / n_total * 100:.0f}%")
print(f"Hard acceptance rate (unanimous include / total)    : {n_unanimous_include / n_total * 100:.0f}%")
print(f"Papers with disagreement                            : {n_disagreement}")
print(f"Papers unanimously excluded                         : {n_unanimous_exclude}")

Total unique papers reviewed                        : 50
Papers with at least one include                    : 30
Papers included by all reviewers who checked it     : 10
Soft acceptance rate (at least one include / total) : 60%
Hard acceptance rate (unanimous include / total)    : 20%
Papers with disagreement                            : 20
Papers unanimously excluded                         : 20


In [10]:
research_areas = list(merged[merged["n_include"] >=1][["MC_Research area", "ME_Research area", "TC_Research area"]].to_numpy().flatten())
research_areas = [i for i in research_areas if pd.notna(i)]
collections.Counter(research_areas)

counts = collections.Counter(research_areas)
total = len(research_areas)

for area, count in counts.most_common():
    print(f"{area}: {count} / {total} ({count / total * 100:.1f}%)")


Polarization: 25 / 54 (46.3%)
Misinformation: 25 / 54 (46.3%)
Both: 4 / 54 (7.4%)


## Round 2

Due to the low agreement in Round 1, we conducted a second round of annotation with a new set of papers. In this case all three annotators reviewed **all 20 papers**. 

In [18]:
iaa_dir = Path("../data/processed/iaa/annotated/round2")

dfs = {a: pd.read_excel(iaa_dir / f"annotator_{a}_annotated.xlsx") for a in ANNOTATORS}

for a, df in dfs.items():
    assert len(df) == 20, f"{a} has {len(df)} rows (expected 40)"
    assert df["Include"].notna().all(), f"{a} has missing Include values"
    print(f"Annotator {a}: {len(df)} rows  ")

Annotator MC: 20 rows  
Annotator ME: 20 rows  
Annotator TC: 20 rows  


## Wide `Include` matrix

One row per paper (20 total), one column per annotator. Since all three annotators reviewed
the same papers, there are no `NaN` values. This backs the agreement metrics and the derived
counts.

In [19]:
long = pd.concat(
    [df[["paper_id", "Include"]].assign(annotator=a) for a, df in dfs.items()],
    ignore_index=True,
)
include_wide = long.pivot(index="paper_id", columns="annotator", values="Include")[ANNOTATORS]
assert include_wide.notna().all().all(), "unexpected NaN: not every annotator reviewed every paper"
assert include_wide.shape == (20, 3), f"unexpected shape {include_wide.shape}"
print(f"Unique papers reviewed: {len(include_wide)}")
include_wide.head()

Unique papers reviewed: 20


annotator,MC,ME,TC
paper_id,,,
0a4d22ac598fbb59,True,True,True
220d0e81790011b7,True,True,True
2c83ac1833318c6e,True,True,True
376761fc4e969e90,True,True,True
41c4eb47dff9837d,False,False,False


## 1. Cohen's kappa per pair of annotators

Each pair shares all **20** papers.

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ the agreement expected by chance from the annotators' marginal label frequencies.

In [20]:
kappa_rows = []
for x, y in combinations(ANNOTATORS, 2):
    both = include_wide[[x, y]].dropna()  # all 20 papers reviewed by both
    k = cohen_kappa_score(both[x].astype(int), both[y].astype(int))
    kappa_rows.append({"pair": f"{x}-{y}", "n_shared": len(both), "cohen_kappa": k})

kappa_df = pd.DataFrame(kappa_rows).round(3)
print(kappa_df[["pair", "cohen_kappa"]])
kappa_df

    pair  cohen_kappa
0  MC-ME          0.9
1  MC-TC          1.0
2  ME-TC          0.9


,pair,n_shared,cohen_kappa
0,MC-ME,20,0.9
1,MC-TC,20,1.0
2,ME-TC,20,0.9


## 2. Krippendorff's alpha (3 annotators)

Computed on all **20 papers** (every annotator reviewed all of them). The reliability
matrix has shape `3 raters × 20 units`.

In [21]:
reliability = include_wide[ANNOTATORS].astype(int).T.values  # 3 x 20

alpha = krippendorff.alpha(reliability_data=reliability, level_of_measurement="nominal")
print(f"Papers reviewed by all three: {len(include_wide)}")
print(f"Krippendorff's alpha (nominal): {alpha:.3f}")

Papers reviewed by all three: 20
Krippendorff's alpha (nominal): 0.934


## 3. Merged annotations table

One row per paper (20 total), carrying the paper metadata once plus every annotator's
`Include`, `To discuss`, `Research area`, `Topic/Ideas` and `Notes` (suffixed by annotator
code). Since all three reviewed every paper, there are no `NaN` verdicts.

Derived columns:
- `n_reviewers` — how many annotators reviewed the paper (always 3 here),
- `n_include` — how many marked it for inclusion,
- `disagreement` — `True` when the reviewers are **not unanimous** (at least one disagrees),
- `any_to_discuss` — `True` when at least one reviewer flagged the paper "To discuss".

In [22]:
META_COLS_R2 = [c for c in META_COLS if c != "iaa_set"]  # round 2 files have no iaa_set

# Paper-level columns once (identical across annotators for a given paper).
meta = (pd.concat([df[["paper_id"] + META_COLS_R2] for df in dfs.values()], ignore_index=True)
        .drop_duplicates(subset="paper_id"))

# Per-annotator annotation columns, suffixed by annotator code.
ann_frames = [
    df[["paper_id"] + ANNOTATION_COLS]
    .rename(columns={c: f"{a}_{c}" for c in ANNOTATION_COLS})
    .set_index("paper_id")
    for a, df in dfs.items()
]
ann_wide = ann_frames[0].join(ann_frames[1:], how="outer")

merged = meta.set_index("paper_id").join(ann_wide, how="outer").reset_index()

# Derived columns from the per-annotator Include verdicts (all 3 reviewers per paper).
inc_cols = [f"{a}_Include" for a in ANNOTATORS]
merged["n_reviewers"] = merged[inc_cols].notna().sum(axis=1)
merged["n_include"] = (merged[inc_cols] == True).sum(axis=1)
merged["disagreement"] = (merged["n_include"] > 0) & (merged["n_include"] < merged["n_reviewers"])

# True if at least one reviewer flagged the paper "To discuss".
disc_cols = [f"{a}_To discuss" for a in ANNOTATORS]
merged["any_to_discuss"] = (merged[disc_cols] == True).any(axis=1)

# Column order: identifiers, metadata, per-annotator Include, derived flags, notes.
note_cols = [f"{a}_{c}" for c in ANNOTATION_COLS if c != "Include" for a in ANNOTATORS]
ordered = (["paper_id"]
           + META_COLS_R2
           + inc_cols
           + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]
           + note_cols)
merged = merged[ordered]

print(f"Merged table: {merged.shape[0]} papers x {merged.shape[1]} columns")
merged[["paper_id", "title"] + inc_cols
       + ["n_reviewers", "n_include", "disagreement", "any_to_discuss"]]

Merged table: 20 papers x 27 columns


,paper_id,title,MC_Include,ME_Include,TC_Include,n_reviewers,n_include,disagreement,any_to_discuss
0,0a4d22ac598fbb59,Analyzing Public Discourse and Sentiment in Cl...,True,True,True,3,3,False,False
1,220d0e81790011b7,Both AI-generated and human influencers can co...,True,True,True,3,3,False,False
2,2c83ac1833318c6e,Apocalyptic Climate Change Conspiracy Theories...,True,True,True,3,3,False,False
3,376761fc4e969e90,Responses to heat waves: what can Twitter data...,True,True,True,3,3,False,True
4,41c4eb47dff9837d,Opposing Yet Still Disseminating Conspiracy Th...,False,False,False,3,0,False,False
5,53f489652a920613,Retrieval of melt ponds on arctic multiyear se...,False,False,False,3,0,False,False
6,5ad9521f3ac24fa9,Can giant surveys of scientists fight misinfor...,False,False,False,3,0,False,False
7,5b3afa534ea45e5b,Augmented CARDS: A machine learning approach t...,False,False,False,3,0,False,False
8,5d035c9013ad404f,The credibility cocktail on Instagram: Assessi...,True,True,True,3,3,False,True
9,5f473c034602c53f,Fake news and indifference to scientific fact:...,True,True,True,3,3,False,False


In [23]:
output_path = iaa_dir / "iaa_merged.xlsx"
merged.to_excel(output_path, index=False)
print(f"Wrote {output_path}")

Wrote ../data/processed/iaa/annotated/round2/iaa_merged.xlsx


## 4. Basic statistics

In [24]:
n_total = len(merged)
n_any_include = int((merged["n_include"] >= 1).sum())
n_unanimous_include = int((merged["n_include"] == merged["n_reviewers"]).sum())
n_disagreement = int(merged["disagreement"].sum())
n_unanimous_exclude = int((merged["n_include"] == 0).sum())

print(f"Total unique papers reviewed                        : {n_total}")
print(f"Papers with at least one include                    : {n_any_include}")
print(f"Papers included by all reviewers who checked it     : {n_unanimous_include}")
print(f"Soft acceptance rate (at least one include / total) : {n_any_include / n_total * 100:.0f}%")
print(f"Hard acceptance rate (unanimous include / total)    : {n_unanimous_include / n_total * 100:.0f}%")
print(f"Papers with disagreement                            : {n_disagreement}")
print(f"Papers unanimously excluded                         : {n_unanimous_exclude}")

Total unique papers reviewed                        : 20
Papers with at least one include                    : 11
Papers included by all reviewers who checked it     : 10
Soft acceptance rate (at least one include / total) : 55%
Hard acceptance rate (unanimous include / total)    : 50%
Papers with disagreement                            : 1
Papers unanimously excluded                         : 9


In [25]:
research_areas = list(merged[merged["n_include"] >= 1][[f"{a}_Research area" for a in ANNOTATORS]].to_numpy().flatten())
research_areas = [i for i in research_areas if pd.notna(i)]

counts = collections.Counter(research_areas)
total = len(research_areas)

for area, count in counts.most_common():
    print(f"{area}: {count} / {total} ({count / total * 100:.1f}%)")

Misinformation: 17 / 32 (53.1%)
Polarazation: 8 / 32 (25.0%)
Both: 7 / 32 (21.9%)
